# Project SD-03 — Excel RAG

> Goal: Build a RAG pipeline over a multi-sheet Excel workbook by
> parsing every data row into a header-attached `Document` — and see
> why the naive way of flattening a spreadsheet to plain text breaks
> row-level retrieval.

This is part of the **Special Documents** series (SD-01…SD-08), where
**parsing** is the changed pipeline block. Everything after the load
step — split, embed, store, retrieve, prompt, answer — is the same
pipeline you already know from Project 01.

```
Parser     : openpyxl (sheet-aware — one Document per data row)
Splitter   : RecursiveCharacterTextSplitter (safety net, large chunks)
Embedding  : Gemini Embedding
Vector DB  : Chroma
Retriever  : Similarity Search (Top-K)
Prompt     : Basic Context + Question
LLM        : Gemini 2.5 Flash
```

Learn:

* Spreadsheet parsing with `openpyxl`
* Header injection as a schema preamble
* Merged cells, column types, and row integrity


### The naive way (what breaks)

An `.xlsx` file is **not** a text file — it is a zip of XML
worksheets laid out as a (sheet, column, row) grid. The tempting
shortcut is to flatten it into text and treat it like a CSV dump.
That shortcut breaks row-level retrieval in three ways:

* **Values are orphaned.** `ws.values` yields raw tuples like
  `(2024-04-15, 'Widget', 25, 19.99, 499.75)` with no column names
  attached — the model sees `25` and `19.99` but has no idea they
  mean `units` and `unit_price`.
* **A text splitter slices mid-record.**
  `RecursiveCharacterTextSplitter` cuts on character/newline
  boundaries it cannot see inside a grid — one chunk ends with
  `unit_price is 19.` and the next starts with `99`, splitting a
  single record across two chunks.
* **Numbers and dates lose context.** A bare `2024-04-15` or `499`
  is meaningless without its header, so exact-value questions
  ("what was the total?") become unanswerable.

Run the two cells below to see it happen — then we build the
correct, sheet-aware parser.


In [ ]:
from openpyxl import load_workbook

EXCEL_PATH = "../../../Data/SD-03-excel/SampleSS.xlsx"
wb = load_workbook(EXCEL_PATH, data_only=True)

# Flatten every sheet into raw text, CSV-dump style.
raw_text = ""
for ws in wb.worksheets:
    raw_text += f"=== sheet: {ws.title} ===\n"
    for row in ws.values:
        cells = [str(v) for v in row if v is not None]
        raw_text += " | ".join(cells) + "\n"

print(raw_text[:600])


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A text splitter has no idea that rows are atomic records.
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=90,
    chunk_overlap=10,
)
naive_chunks = naive_splitter.split_text(raw_text)

print(f"flattened text split into {len(naive_chunks)} chunks")
print("--- chunk 1: a record sliced mid-way ---")
print(naive_chunks[1])


## 0 · Setup — environment & keys

**WHAT:** Loads `.env` (so the Gemini API key is available) and
imports the same LangChain stack as the baseline project. No install
cell is needed here — `openpyxl` is already in `requirements.txt`.

**WHY:** One shared place for keys and imports keeps every pipeline
cell below short and readable. The only library that is new compared
with Project 01 is `openpyxl`, the standard Python reader/writer for
`.xlsx` files.

**WHAT TO EXPECT:** The key cell prints `True` from `load_dotenv()`
and a masked key prefix (`abcd…`) when `GOOGLE_API_KEY` is present,
or a friendly message telling you to create a `.env` file.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if api_key:
    print(f"GOOGLE_API_KEY set: {api_key[:4]}…")
else:
    print("No GOOGLE_API_KEY found — copy .env.example to .env and add yours.")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from openpyxl import load_workbook


## 1 · Load — parse the workbook sheet-by-sheet

**WHAT:** `openpyxl` opens the workbook and we iterate the sheets in
order. The sample, `Data/SD-03-excel/SampleSS.xlsx`, is a
**multi-sheet** workbook:

| Sheet | Contents |
|-------|----------|
| Sales | sales ledger — `date`, `product`, `units`, `unit_price`, `total` |
| Employees | directory — `id`, `name`, `department`, `joining_date` |
| Products | catalog — `SKU`, `name`, `category`, `stock` |

**WHY:** Unlike the text loaders you have seen, a spreadsheet has **no
natural reading order** — meaning lives in the (sheet, column header,
row) grid. So instead of one big document per file, we make **one
`Document` per data row**, injecting the column headers as a schema
preamble:

```
Sales: date is 2024-04-15, product is Widget, units is 25, unit_price is 19.99, total is 499.75
```

Every value stays attached to its header, and `metadata` records
`sheet_name` and `row` so any retrieved chunk can be traced back to
its exact cell. The parser is written **data-adaptively** — it reads
whatever headers it finds, so it works on any workbook that has a
header row.

**WHAT TO EXPECT:** The list of sheets, then `N` row-documents whose
`page_content` starts with `<sheet>: <col> is <value>, ...`.


In [ ]:
EXCEL_PATH = "../../../Data/SD-03-excel/SampleSS.xlsx"

if not os.path.exists(EXCEL_PATH):
    print(f"Sample workbook not found: {EXCEL_PATH}")
    print("Place SampleSS.xlsx under Data/SD-03-excel/.")
else:
    wb = load_workbook(EXCEL_PATH, data_only=True)
    print("Sheets:", wb.sheetnames)
    for ws in wb.worksheets:
        print(f"  {ws.title}: {ws.max_row} rows x {ws.max_column} cols")


In [ ]:
def cell_value(ws, cell):
    """Return the cell value, resolving merged cells to the top-left value."""
    if cell.value is not None:
        return cell.value
    for mrange in ws.merged_cells.ranges:
        if (mrange.min_row <= cell.row <= mrange.max_row and
                mrange.min_col <= cell.column <= mrange.max_col):
            return ws.cell(mrange.min_row, mrange.min_col).value
    return None


In [ ]:
def row_to_text(ws, header, row):
    """Join a data row's values to their column headers as a preamble."""
    parts = []
    for name, cell in zip(header, row):
        value = cell_value(ws, cell)
        if value is not None:
            parts.append(f"{name} is {value}")
    return ", ".join(parts)


In [ ]:
def sheet_documents(ws):
    """Build the row-Documents for one worksheet."""
    rows = list(ws.iter_rows())
    if not rows:
        return []
    header = [str(c.value) if c.value is not None else f"col_{i+1}"
              for i, c in enumerate(rows[0])]
    result = []
    for r_idx, row in enumerate(rows[1:], start=2):
        text = row_to_text(ws, header, row)
        if text:
            result.append(Document(page_content=f"{ws.title}: {text}",
                                  metadata={"sheet_name": ws.title, "row": r_idx}))
    return result


In [ ]:
def parse_workbook(path):
    """Parse every sheet of a workbook into row-documents."""
    wb = load_workbook(path, data_only=True)
    documents = []
    for ws in wb.worksheets:
        documents.extend(sheet_documents(ws))
    return documents


In [ ]:
docs = parse_workbook(EXCEL_PATH)
print(f"parsed {len(docs)} row-documents")

print("--- first document ---")
print(docs[0].page_content)
print(docs[0].metadata)


## 1b · Column-type tagging (optional)

**WHAT:** A small helper that classifies each column as `text`,
`number`, or `date` by looking at the values already in the
workbook.

**WHY:** Spreadsheets silently mix types. A bare `datetime` or `int`
reads as meaningless noise once it is flattened to text — tagging
records the *kind* of data each header holds, which helps a later
LLM interpret values correctly and stops dates from being confused
with numbers.

**WHAT TO EXPECT:** A per-sheet dictionary such as
`{'date': 'date', 'product': 'text', 'units': 'number', ...}`.


In [ ]:
import datetime

def infer_column_type(values):
    """Classify a column as 'date', 'number', or 'text'."""
    non_null = [v for v in values if v is not None]
    if not non_null:
        return "empty"
    if all(isinstance(v, (datetime.datetime, datetime.date)) for v in non_null):
        return "date"
    if all(isinstance(v, (int, float)) for v in non_null):
        return "number"
    return "text"


In [ ]:
for ws in wb.worksheets:
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        continue
    header = [str(h) if h is not None else f"col_{i+1}" for i, h in enumerate(rows[0])]
    types = {}
    for col_idx, name in enumerate(header):
        values = [r[col_idx] for r in rows[1:] if col_idx < len(r)]
        types[name] = infer_column_type(values)
    print(f"{ws.title}: {types}")


## 2 · Split — rows are already atomic

**WHAT:** We still run `RecursiveCharacterTextSplitter`, but only as a
safety net: `chunk_size=1000` (larger than any row-document) with
`chunk_overlap=0`.

**WHY:** This is a *parsing* project — the splitter is **not** the
changed block. Each `Document` is already one atomic record with its
headers attached, so splitting it would re-create the exact
mid-record corruption we saw in the naive demo. A huge `chunk_size`
guarantees every row-document stays a single chunk, and
`chunk_overlap=0` stops one record's tail from bleeding into the
next.

**WHAT TO EXPECT:** The chunk count equals the row-document count —
the splitter changed nothing, and every chunk keeps its metadata.


In [ ]:
safety_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
)

chunks = safety_splitter.split_documents(docs)
print(f"{len(chunks)} chunks from {len(docs)} row-documents")

print("--- chunk 0 keeps its preamble + metadata ---")
print(chunks[0].page_content)
print(chunks[0].metadata)


## 3 · Embed — rows with headers become meaningful vectors

**WHAT:** `GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")`
creates the embedding model used to vectorize every row-document, and
the next cell embeds one sample chunk to show the dimensionality.

**WHY:** Embeddings put *similar meaning* near each other in vector
space. A preamble that says `Sales: date is 2024-04-15, product is
Widget, ...` embeds far better than a flat
`2024-04-15 | Widget | 25 | 19.99 | 499.75` — the header words give
the vector its semantic anchors, which is exactly what row-integrity
retrieval depends on.

**WHAT TO EXPECT:** An embeddings object, then a printed vector
dimension and its first few floats.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")


In [ ]:
sample_vector = embeddings.embed_query(chunks[0].page_content)
print(f"embedding dimension: {len(sample_vector)}")
print(sample_vector[:5])


## 4 · Store — index the row-documents in Chroma

**WHAT:** `Chroma.from_documents(...)` embeds every chunk and indexes
the vectors in a Chroma collection (a `chroma_langchain_db/` folder
is created next to the notebook).

**WHY:** The vector store is the pipeline's memory — it lets the
retriever find the relevant row-documents in milliseconds. Because
our metadata keeps `sheet_name` and `row`, every stored vector knows
exactly which cell it came from.

**WHAT TO EXPECT:** A `Chroma` object assigned to `vector_store`.


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 5 · Retrieve — find the exact row-documents for a question

**WHAT:** `vector_store.similarity_search(query, k=3)` returns the 3
row-documents most similar to the question, and the next cell prints
each result with its similarity score and source row.

**WHY:** This is the "R" in RAG — evidence gathering. The query
below is an **exact-value retrieval** question: it can only be
answered if a single intact row-document carries both the product
name and its `total` value together. With the naive flattening, the
value would be orphaned from its header and this question would be
unanswerable.

**WHAT TO EXPECT:** 3 row-documents, each tagged with
`sheet=<Sales> row=<n>` and a similarity score.


In [ ]:
query = "What was the total sales for product X in Q2?"

retrieved = vector_store.similarity_search(query, k=3)
print(f"retrieved {len(retrieved)} documents")


In [ ]:
scored = vector_store.similarity_search_with_score(query, k=3)
for doc, score in scored:
    sheet = doc.metadata["sheet_name"]
    row = doc.metadata["row"]
    print(f"score={score:.4f}  {sheet} row {row}")
    print(doc.page_content)
    print()


## 6 · Prompt — package the row-documents as context

**WHAT:** A `ChatPromptTemplate` wraps the "answer using ONLY the
provided context" instruction around the `context` and `question`
slots. The next cell renders the prompt so you can read exactly what
the model receives.

**WHY:** The prompt is the anti-hallucination contract: the model may
*only* answer from the retrieved row-documents. Because each row
keeps its headers, the model sees *structured records*, not detached
fragments — which is what makes exact-value questions answerable.

**WHAT TO EXPECT:** A `ChatPromptTemplate`, then printed `messages`
containing the instructions, the retrieved preambles, and the
question.


In [ ]:
template = """
You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not contained in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
context = "\n\n".join(doc.page_content for doc in retrieved)

messages = prompt.invoke({
    "context": context,
    "question": query,
})
print(messages)


## 7 · Answer — the LLM reads the prompt

**WHAT:** `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` invokes
the filled `messages` and the answer is printed.

**WHY:** This is the final block: the model reads the retrieved
evidence plus the question and produces a grounded answer. If the
workbook really contains a sales row for "product X" with its `total`
value, the answer is copied from the context — not guessed from the
model's memory.

**WHAT TO EXPECT:** A natural-language answer drawn from the
retrieved row-documents.


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke(messages)
print(response.content)


## 8 · Try it yourself — your sandbox

Change `query` below, tweak `k`, or ask a question that spans two
sheets. The second cell runs the employee-sheet question from the
intro end to end.

> Tip: exact-value questions ("how many units?", "who joined in
> June?") are the sharpest test of row integrity — if the answer
> comes back garbled or mixes two rows, the parser is the place to
> look.


In [ ]:
query2 = "Who joined the company in June?"

retrieved2 = vector_store.similarity_search(query2, k=3)
for doc in retrieved2:
    print(doc.page_content)


In [ ]:
context2 = "\n\n".join(doc.page_content for doc in retrieved2)
messages2 = prompt.invoke({
    "context": context2,
    "question": query2,
})

print(llm.invoke(messages2).content)


## What you should notice

- **Parsing is the changed block.** The load step decided everything:
  row-aware openpyxl parsing with header injection made exact-value
  questions answerable; the naive CSV-style flattening made them
  impossible.
- **Headers are semantic anchors.** Writing `product is Widget, total
  is 499.75` instead of `Widget, 499.75` is what lets both the
  embedder and the LLM interpret the row.
- **Rows are atomic — never split them.** Because each row-document is
  already a record, the splitter must be a safety net (huge
  `chunk_size`, `chunk_overlap=0`), not a row slicer.
- **Merged cells hide values.** A merged cell returns `None` unless
  you resolve it to the top-left cell — forgetting this silently
  drops data.
- **Type tags add context.** A bare `datetime` or `int` reads as noise
  after flattening; knowing a column is a date, number, or text helps
  interpretation.
- **Metadata makes retrieval traceable.** `sheet_name` + `row` on every
  `Document` mean each retrieved chunk points back to its exact cell.


## Exercises

1. **Scope retrieval with metadata.** Use Chroma's `where` filter to
   retrieve only from the Sales sheet (`similarity_search` with
   `filter={"sheet_name": "Sales"}`) and re-run the Q2 question —
   notice how the context stays on one sheet.
2. **Handle headerless / shifted tables.** The parser assumes row 1 is
   the header. Extend it to scan for the first fully non-empty row and
   treat that as the header — real sheets often start with a title
   row.
3. **Compare against the CSV loader.** Export one sheet to CSV and
   build the same pipeline with `CSVLoader`. Compare the resulting
   `Document` content and metadata with the openpyxl parser — which
   one keeps more of the spreadsheet's structure?
